<a href="https://colab.research.google.com/github/kartik815/Amazon-ML-Challenge-2026/blob/main/notebooks/04_Blocking_rebuild.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Rebuild Blocking Indexes (condensed)

This replaces the long exploratory version. It rebuilds exactly the 5 production
index variables (`name_index`, `token_index`, `address_token_index`, `char3_index`,
`rare_two_token_index_10`) plus the `missed_s2_ids` set they depend on, with no
experiments/threshold sweeps/transliteration analysis in between. Results are the
same as before — the ≤10 rare two-token-pair blocker gives ~91.4666% blocking recall.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

import os
import gc
import pandas as pd

DRIVE_ROOT = "/content/drive/MyDrive/Amazon ML Challenge 2026"

CLEANED_DATA_ROOT = os.path.join(
    DRIVE_ROOT,
    "03_Experiments",
    "Cleaned_Data"
)

TRAIN_ROOT = os.path.join(
    DRIVE_ROOT,
    "01_Dataset",
    "6ab10eb3b23ba_student_resource",
    "student_resource",
    "dataset",
    "train"
)

TEST_ROOT = os.path.join(
    DRIVE_ROOT,
    "01_Dataset",
    "6ab10eb3b23ba_student_resource",
    "student_resource",
    "dataset",
    "test"
)

BLOCKING_OUTPUT_ROOT = os.path.join(
    DRIVE_ROOT,
    "05_Outputs",
    "Candidate_Pairs"
)

os.makedirs(BLOCKING_OUTPUT_ROOT, exist_ok=True)

print("Cleaned data:")
print(CLEANED_DATA_ROOT)

print("\nBlocking output:")
print(BLOCKING_OUTPUT_ROOT)


Mounted at /content/drive
Cleaned data:
/content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data

Blocking output:
/content/drive/MyDrive/Amazon ML Challenge 2026/05_Outputs/Candidate_Pairs


In [2]:
RAW_S1_PATH = os.path.join(
    TRAIN_ROOT,
    "train_source1.tsv"
)

RAW_S2_PATH = os.path.join(
    TRAIN_ROOT,
    "train_source2.tsv"
)


In [3]:
GT_PATH = os.path.join(
    TRAIN_ROOT,
    "train_ground_truth.tsv"
)

S1_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s1_cleaned.tsv"
)

S2_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s2_cleaned.tsv"
)

gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype={
        "source1_entity_id": "string",
        "matched_entity_ids": "string"
    }
)

print("Ground truth rows:", len(gt))


Ground truth rows: 2206821


In [4]:
gt_lookup = {}

for row in gt.itertuples(index=False):
    s1_id = row.source1_entity_id
    matched = row.matched_entity_ids

    if pd.isna(matched) or matched == "":
        gt_lookup[s1_id] = set()
    else:
        gt_lookup[s1_id] = {
            x.strip()
            for x in str(matched).split(",")
            if x.strip()
        }

print("Ground truth lookup entries:", len(gt_lookup))


Ground truth lookup entries: 2206821


In [5]:
s1_blocking = pd.read_csv(
    S1_PATH,
    sep="\t",
    usecols=[
        "entity_id",
        "country_norm",
        "name_core",
        "address_norm"
    ],
    dtype={
        "entity_id": "string",
        "country_norm": "string",
        "name_core": "string",
        "address_norm": "string"
    }
)

print(s1_blocking.shape)


(2206821, 4)


In [6]:
NAME_STOPWORDS = {
    "inc", "incorporated", "llc", "ltd", "limited",
    "pvt", "private", "company", "co", "corp",
    "corporation", "group", "services", "partners",
    "holdings", "associates", "center", "india",
    "and", "of"
}

def meaningful_tokens(name):
    if pd.isna(name) or not str(name).strip():
        return []

    tokens = str(name).split()

    return [
        token
        for token in tokens
        if token not in NAME_STOPWORDS and len(token) >= 2
    ]


### 1. Exact country + name index (`name_index`)

In [7]:
from collections import defaultdict

def build_name_index(path):
    index = defaultdict(list)

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "country_norm", "name_core"],
        chunksize=200_000,
        dtype={
            "entity_id": "string",
            "country_norm": "string",
            "name_core": "string"
        }
    ):
        chunk = chunk[
            chunk["name_core"].notna() &
            (chunk["name_core"] != "")
        ]

        for country, name, entity_id in zip(
            chunk["country_norm"],
            chunk["name_core"],
            chunk["entity_id"]
        ):
            index[(country, name)].append(entity_id)

        del chunk
        gc.collect()

    return index

print("Building S2 exact-name index...")
name_index = build_name_index(S2_PATH)
print("name_index:", len(name_index))


Building S2 exact-name index...
name_index: 3597772


### 2. Name-token index (`token_index`)

In [8]:
from collections import Counter

name_token_counter = Counter()

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["name_core"],
    chunksize=200_000,
    dtype={"name_core": "string"}
):
    for name in chunk["name_core"].dropna():
        name_token_counter.update(set(meaningful_tokens(name)))

    del chunk
    gc.collect()

allowed_name_tokens = {
    token
    for token, freq in name_token_counter.items()
    if 2 <= freq <= 500
}

print("Unique name tokens:", len(name_token_counter))
print("Allowed name tokens:", len(allowed_name_tokens))

del name_token_counter
gc.collect()


Unique name tokens: 810209
Allowed name tokens: 167295


0

In [9]:
token_index = defaultdict(list)

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):
    for entity_id, name in zip(chunk["entity_id"], chunk["name_core"]):
        if pd.isna(name):
            continue

        for token in set(meaningful_tokens(name)):
            if token in allowed_name_tokens:
                token_index[token].append(entity_id)

    del chunk
    gc.collect()

print("token_index:", len(token_index))


token_index: 167295


### 3. Address-token index (`address_token_index`)

In [10]:
address_token_counts = Counter()

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["address_norm"],
    chunksize=200_000,
    dtype={"address_norm": "string"}
):
    for address in chunk["address_norm"].dropna():
        for token in set(str(address).split()):
            if token:
                address_token_counts[token] += 1
    del chunk
    gc.collect()

MIN_ADDRESS_TOKEN_FREQ = 2
MAX_ADDRESS_TOKEN_FREQ = 500

allowed_address_tokens = {
    token
    for token, count in address_token_counts.items()
    if MIN_ADDRESS_TOKEN_FREQ <= count <= MAX_ADDRESS_TOKEN_FREQ
}

print("Unique address tokens:", len(address_token_counts))
print("Allowed address tokens:", len(allowed_address_tokens))

del address_token_counts
gc.collect()


Unique address tokens: 607900
Allowed address tokens: 404250


0

In [11]:
address_token_index = defaultdict(list)

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["entity_id", "address_norm"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "address_norm": "string"
    }
):
    for entity_id, address in zip(chunk["entity_id"], chunk["address_norm"]):
        if pd.isna(address):
            continue

        for token in set(str(address).split()):
            if token in allowed_address_tokens:
                address_token_index[token].append(entity_id)

    del chunk
    gc.collect()

print("address_token_index:", len(address_token_index))


address_token_index: 404250


### 4. `missed_s2_ids` — true S2 matches not caught by the 3 blockers above

In [12]:
missed_s2_ids = set()

total_true = 0
captured_existing = 0

for row in s1_blocking.itertuples(index=False):
    s1_id = row.entity_id
    country = row.country_norm
    name = row.name_core
    address = row.address_norm

    true_ids = {
        x
        for x in gt_lookup.get(s1_id, set())
        if x.startswith("S2-")
    }

    if not true_ids:
        continue

    total_true += len(true_ids)

    candidate_set = set()

    if pd.notna(name) and name != "":
        candidate_set.update(name_index.get((country, name), []))

        for token in set(meaningful_tokens(name)):
            if token in allowed_name_tokens:
                candidate_set.update(token_index.get(token, []))

    if pd.notna(address) and address != "":
        for token in set(str(address).split()):
            if token in allowed_address_tokens:
                candidate_set.update(address_token_index.get(token, []))

    captured = true_ids.intersection(candidate_set)

    captured_existing += len(captured)

    missed_s2_ids.update(true_ids - captured)

print("Total true S2 matches:", total_true)
print("Captured by existing blocker:", captured_existing)
print("Missed by existing blocker:", len(missed_s2_ids))


Total true S2 matches: 3693619
Captured by existing blocker: 3222405
Missed by existing blocker: 471214


### 5. Character 3-gram index (`char3_index`) — conservative version, freq 2–500

In [13]:
def char_ngrams(text, n=3):
    if pd.isna(text):
        return set()

    text = str(text).replace(" ", "")

    if len(text) < n:
        return {text}

    return {
        text[i:i+n]
        for i in range(len(text) - n + 1)
    }

char3_counter = Counter()

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["name_core"],
    chunksize=200_000,
    dtype={"name_core": "string"}
):
    for name in chunk["name_core"].dropna():
        char3_counter.update(char_ngrams(name, n=3))
    del chunk
    gc.collect()

allowed_char3 = {
    gram
    for gram, freq in char3_counter.items()
    if 2 <= freq <= 500
}

print("Unique 3-grams:", len(char3_counter))
print("Allowed 3-grams:", len(allowed_char3))

del char3_counter
gc.collect()


Unique 3-grams: 90023
Allowed 3-grams: 65473


0

In [14]:
char3_index = defaultdict(list)

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):
    for entity_id, name in zip(chunk["entity_id"], chunk["name_core"]):
        if pd.isna(name):
            continue

        for gram in char_ngrams(name, n=3):
            if gram in allowed_char3:
                char3_index[gram].append(entity_id)

    del chunk
    gc.collect()

print("char3_index:", len(char3_index))


char3_index: 65473


### 6. Rare two-token-pair index (`rare_two_token_index_10`), freq ≤ 10

In [15]:
missed_s2_names = {}

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):
    mask = chunk["entity_id"].isin(missed_s2_ids)

    for entity_id, name in zip(
        chunk.loc[mask, "entity_id"],
        chunk.loc[mask, "name_core"]
    ):
        missed_s2_names[entity_id] = name

    del chunk
    gc.collect()

print("Missed S2 names:", len(missed_s2_names))


Missed S2 names: 471214


In [16]:
from itertools import combinations

def name_token_pairs(name):
    tokens = sorted(set(meaningful_tokens(name)))

    if len(tokens) < 2:
        return set()

    return set(combinations(tokens, 2))

two_token_frequency = defaultdict(int)

for name in missed_s2_names.values():
    if pd.isna(name):
        continue
    for pair in name_token_pairs(name):
        two_token_frequency[pair] += 1

print("Unique two-token pairs:", len(two_token_frequency))


Unique two-token pairs: 362476


In [17]:
rare_two_token_index_10 = defaultdict(list)

for s2_id, name in missed_s2_names.items():
    if pd.isna(name):
        continue

    for pair in name_token_pairs(name):
        if two_token_frequency[pair] <= 10:
            rare_two_token_index_10[pair].append(s2_id)

print("rare_two_token_index_10:", len(rare_two_token_index_10))
print(
    "Indexed references:",
    sum(len(v) for v in rare_two_token_index_10.values())
)


rare_two_token_index_10: 353966
Indexed references: 522162


### Verify everything

In [18]:
print("name_index:", len(name_index))
print("token_index:", len(token_index))
print("address_token_index:", len(address_token_index))
print("char3_index:", len(char3_index))
print("rare_two_token_index_10:", len(rare_two_token_index_10))


name_index: 3597772
token_index: 167295
address_token_index: 404250
char3_index: 65473
rare_two_token_index_10: 353966


In [19]:
import os
import gc
import pandas as pd

CANDIDATE_OUTPUT_DIR = os.path.join(
    DRIVE_ROOT,
    "05_Outputs",
    "Candidate_Pairs"
)

os.makedirs(CANDIDATE_OUTPUT_DIR, exist_ok=True)

CANDIDATE_PATH = os.path.join(
    CANDIDATE_OUTPUT_DIR,
    "train_candidate_pairs.tsv"
)

# Remove an old partial output if it exists
if os.path.exists(CANDIDATE_PATH):
    os.remove(CANDIDATE_PATH)

print("Output:", CANDIDATE_PATH)

Output: /content/drive/MyDrive/Amazon ML Challenge 2026/05_Outputs/Candidate_Pairs/train_candidate_pairs.tsv


In [ ]:
# =========================================================
# PRODUCTION CANDIDATE GENERATION
# =========================================================

import os
import gc
import pandas as pd

S1_CHUNK_SIZE = 50_000

first_write = True
total_candidate_pairs = 0
total_s1_processed = 0

for chunk_no, chunk in enumerate(
    pd.read_csv(
        S1_PATH,
        sep="\t",
        usecols=[
            "entity_id",
            "country_norm",
            "name_core",
            "address_norm"
        ],
        chunksize=S1_CHUNK_SIZE,
        dtype={
            "entity_id": "string",
            "country_norm": "string",
            "name_core": "string",
            "address_norm": "string"
        }
    ),
    start=1
):

    candidate_rows = []

    for row in chunk.itertuples(index=False):

        s1_id = row.entity_id
        country = row.country_norm
        name = row.name_core
        address = row.address_norm

        candidates = set()

        # -------------------------------------------------
        # 1. Exact country + name
        # -------------------------------------------------
        if pd.notna(country) and pd.notna(name):
            candidates.update(
                name_index.get((country, name), [])
            )

        # -------------------------------------------------
        # 2. Name token blocker
        # -------------------------------------------------
        if pd.notna(name):

            for token in meaningful_tokens(name):

                if token in allowed_name_tokens:
                    candidates.update(
                        token_index.get(token, [])
                    )

        # -------------------------------------------------
        # 3. Address token blocker
        # -------------------------------------------------
        if pd.notna(address):

            for token in set(str(address).split()):

                if token in allowed_address_tokens:
                    candidates.update(
                        address_token_index.get(token, [])
                    )

        # -------------------------------------------------
        # 4. Character 3-gram blocker
        # -------------------------------------------------
        if pd.notna(name):

            for gram in char_ngrams(name, n=3):

                if gram in allowed_char3:
                    candidates.update(
                        char3_index.get(gram, [])
                    )

        # -------------------------------------------------
        # 5. Rare two-token pair blocker
        #    frequency <= 10
        # -------------------------------------------------
        if pd.notna(name):

            for pair in name_token_pairs(name):

                candidates.update(
                    rare_two_token_index_10.get(pair, [])
                )

        # -------------------------------------------------
        # Store candidate pairs
        # -------------------------------------------------
        for s2_id in candidates:

            candidate_rows.append(
                (s1_id, s2_id)
            )

    # -----------------------------------------------------
    # Write this chunk immediately
    # -----------------------------------------------------
    if candidate_rows:

        candidate_df = pd.DataFrame(
            candidate_rows,
            columns=[
                "source1_entity_id",
                "source2_entity_id"
            ]
        )

        # Defensive deduplication
        candidate_df.drop_duplicates(
            inplace=True
        )

        candidate_df.to_csv(
            CANDIDATE_PATH,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )

        total_candidate_pairs += len(candidate_df)

        del candidate_df
        del candidate_rows

    total_s1_processed += len(chunk)

    if chunk_no % 5 == 0:

        print(
            f"Processed S1: {total_s1_processed:,} | "
            f"Candidate pairs written: "
            f"{total_candidate_pairs:,}"
        )

    del chunk
    gc.collect()

    first_write = False

print("\n========================================")
print("CANDIDATE GENERATION COMPLETE")
print("========================================")
print(
    "S1 records processed:",
    f"{total_s1_processed:,}"
)
print(
    "Candidate pairs written:",
    f"{total_candidate_pairs:,}"
)
print("Output:", CANDIDATE_PATH)